# Inspect UA Daily 4-km SWE / Snow Depth — post-consolidator review

Per-source inspection of the **University of Arizona Daily 4-km Gridded SWE and Snow Depth** product (NSIDC-0719 v1, Broxton, Zeng & Dawson, DOI [10.5067/0GGPB220EX6A](https://nsidc.org/data/nsidc-0719/versions/1)), as it lands in this repo's consolidated form at `<datastore>/ua_swe/daily/ua_swe_daily_WY<YYYY>.nc`.

This is the companion to the cross-source views in [consolidated/inspect_consolidated_swe.ipynb](consolidated/inspect_consolidated_swe.ipynb) and [consolidated/inspect_consolidated_snow_covered_area.ipynb](consolidated/inspect_consolidated_snow_covered_area.ipynb), focused on validating UA SWE in isolation **before** committing to the aggregate / target wiring (PR-B → PR-D).

**What the consolidator does** (PR-A `fetch/ua_swe.py`):

1. Download the per-WY native NetCDF from NSIDC HTTPS (Earthdata-authenticated).
2. Open, decode time as `days since 1900-01-01`, rename `SWE` → `swe` and `DEPTH` → `snow_depth`.
3. Per-day reproject from NAD83 EPSG:4269 lat/lon to EPSG:5070 4 km (NN resampling, dst grid locked from day 0).
4. Apply CF-1.6 metadata via `fetch.consolidate.apply_cf_metadata` (units, `long_name`, `cell_methods: "time: point"`, `grid_mapping: crs`).
5. Atomic write with zlib=4 chunked encoding.

**What we check here:**

- Schema, dims, time coverage, CRS attachment.
- A peak-winter day map (CONUS) for both `swe` and `snow_depth` to confirm spatial pattern + magnitude.
- A NaN-coverage map so we can see what fraction of CONUS is valid.
- The depth-derived **binary snow-cover preview** that PR-B will area-weight into a per-HRU fractional SCA.
- A WY seasonal cycle for one Western US sample box (CONUS-domain SWE peaks in late Feb / early Mar).
- A magnitude sanity-check against published Broxton CONUS climatology.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

DATASTORE = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/nhf-datastore")
DAILY_DIR = DATASTORE / "ua_swe" / "daily"

# WY filename convention. For TARGET_DATE in March, the right WY is the one
# ending in TARGET_DATE.year (water year = Oct(YR-1) -> Sep(YR)).
TARGET_DATE = "2010-03-01"
TARGET_WY = 2010

DEPTH_THRESHOLD_MM = 1.0  # provisional PR-D default for binary SCA preview

## 1. Schema, time coverage, and water-year inventory

List every WY that landed on disk, then open one of them to confirm the variable layout, CRS attachment, and CF metadata.

In [ ]:
wy_files = sorted(DAILY_DIR.glob("ua_swe_daily_WY*.nc"))
print(f"Water-year files on disk: {len(wy_files)}")
if wy_files:
    wys = sorted(int(p.stem.split("WY")[1]) for p in wy_files)
    print(f"  WY range: {wys[0]} - {wys[-1]}")
    gaps = [wy for wy in range(wys[0], wys[-1] + 1) if wy not in wys]
    print(f"  Gaps in range: {gaps if gaps else 'none'}")

target_path = DAILY_DIR / f"ua_swe_daily_WY{TARGET_WY}.nc"
if not target_path.exists():
    raise FileNotFoundError(f"Missing {target_path}; run `pixi run nhf-targets fetch ua-swe`.")

ds = xr.open_dataset(target_path)
print()
print(f"=== {target_path.name} ===")
print(ds)

**Expected schema** (per PR-A `catalog/sources.yml` and the consolidator):

| Var | Type | Units | long_name | cell_methods |
| --- | ---- | ----- | --------- | ------------ |
| `swe` | float32 | `kg m-2` | snow water equivalent | `time: point` |
| `snow_depth` | float32 | `mm` | snow depth | `time: point` |
| `crs` | int (scalar) | — | EPSG:5070 NAD83 / Conus Albers (`grid_mapping_name`, `spatial_ref` WKT) | — |

| Dim | Length | Notes |
| --- | ------ | ----- |
| `time` | ~365 (366 in leap years) | `time: mean` is **wrong** for daily snapshots; the consolidator deliberately writes `time: point`. |
| `y` | 802 | EPSG:5070 north-up (decreasing y values) |
| `x` | 1488 | EPSG:5070 west-to-east (increasing x values) |

A schema mismatch here means the consolidator drifted from spec — diagnose by reading the source NSIDC file directly and comparing.

In [ ]:
# Quick assertions on the schema; raise loudly if anything has drifted.
assert "swe" in ds.data_vars, "swe variable missing"
assert "snow_depth" in ds.data_vars, "snow_depth variable missing"
assert "crs" in ds.data_vars, "crs grid-mapping variable missing"
assert ds["swe"].attrs.get("units") == "kg m-2", f"swe units = {ds['swe'].attrs.get('units')!r}"
assert ds["snow_depth"].attrs.get("units") == "mm", f"snow_depth units = {ds['snow_depth'].attrs.get('units')!r}"
assert ds["swe"].attrs.get("cell_methods") == "time: point"
assert ds["swe"].attrs.get("grid_mapping") == "crs"
print("Schema OK.")
print()
print(f"time:     {ds.time.values[0]} -> {ds.time.values[-1]} ({ds.sizes['time']} steps)")
print(f"y:        {float(ds.y.values[0]):.0f} -> {float(ds.y.values[-1]):.0f} m EPSG:5070  ({ds.sizes['y']} cells)")
print(f"x:        {float(ds.x.values[0]):.0f} -> {float(ds.x.values[-1]):.0f} m EPSG:5070  ({ds.sizes['x']} cells)")
print(f"crs.grid_mapping_name = {ds['crs'].attrs.get('grid_mapping_name')!r}")
print(f"crs.spatial_ref start = {ds['crs'].attrs.get('spatial_ref','')[:80]!r}...")

## 2. Peak-winter day — SWE and snow depth panels

CONUS snowpack peaks in late February / early March; March 1 is a good universal date for the magnitude check. Both variables share the same NaN footprint (the consolidator masks fill `< 0` for both pre-projection).

In [ ]:
target_day = ds.sel(time=TARGET_DATE, method="nearest")
actual_day = str(target_day.time.values)[:10]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Robust colour scaling: use 98th percentile of non-NaN pixels so the deep
# Sierra Nevada doesn't wash out the rest of the colour ramp.
for ax, var, cmap in [(axes[0], "swe", "Blues"), (axes[1], "snow_depth", "Blues")]:
    da = target_day[var]
    finite = da.values[~np.isnan(da.values)]
    if finite.size == 0:
        ax.set_title(f"{var} — all-NaN day")
        continue
    vmax = float(np.percentile(finite, 98))
    da.plot(
        ax=ax, cmap=cmap, vmin=0, vmax=vmax,
        cbar_kwargs={"orientation": "horizontal", "shrink": 0.8, "pad": 0.08, "aspect": 30, "label": f"{var} ({da.attrs.get('units','')})"},
    )
    ax.set_title(f"UA SWE WY{TARGET_WY} | {var} | {actual_day}\nrobust scale: 0 - {vmax:.0f} {da.attrs.get('units','')}", fontsize=11)
    ax.set_xlabel("x (m, EPSG:5070)")
    ax.set_ylabel("y (m, EPSG:5070)")
    ax.set_aspect("equal")

fig.suptitle("UA daily 4-km SWE & snow depth — peak winter CONUS view", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**What to look for:**

- Deepest snow over the Sierra Nevada, Cascades, northern Rockies, and Adirondacks.
- A thin snowpack band across the Northern Plains.
- No snow south of ~35° N (only the Sierra Nevada extends south past that).
- `swe` ≈ `snow_depth / 4` in mid-winter (density ~250 kg/m³ → 4 mm depth / 1 mm SWE) for a quick magnitude sanity check.

**Red flags:** uniform colour across CONUS (fill-value decoding bug), no Sierra signal (wrong-day or unit error), values >2000 mm SWE on a CONUS scale (decimal-point shift).

## 3. NaN-coverage and CRS footprint

What fraction of the grid carries valid data? UA SWE's CONUS footprint excludes ocean and the NSIDC mask. A coverage map per pixel ("is any time step finite") visualizes the actual CONUS land footprint that the aggregator will see.

In [ ]:
coverage = ds["swe"].notnull().any(dim="time")
finite_frac = float(coverage.sum().values) / float(coverage.size)
print(f"Per-pixel any-finite coverage across WY{TARGET_WY}: {finite_frac:.1%} of grid cells")

fig, ax = plt.subplots(figsize=(10, 7))
coverage.astype("int8").plot(ax=ax, cmap="Greys", add_colorbar=True, cbar_kwargs={"orientation": "horizontal", "shrink": 0.6, "pad": 0.08, "aspect": 30, "label": "any-time-step finite (0/1)"})
ax.set_title(f"UA SWE WY{TARGET_WY} — CONUS land footprint (any-time-step finite mask)", fontsize=11)
ax.set_xlabel("x (m, EPSG:5070)")
ax.set_ylabel("y (m, EPSG:5070)")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 4. Depth-derived snow-cover preview (PR-B aggregator preview)

This is what the PR-B aggregator will compute pre-aggregation: per-pixel `snow_depth > threshold_mm` → 0/1 → area-weighted to HRUs by gdptools = fractional snow-covered area per HRU. The pixel-level binary is shown here as a sanity-check before any aggregation.

In [ ]:
depth_day = target_day["snow_depth"]
binary = (depth_day > DEPTH_THRESHOLD_MM).where(depth_day.notnull()).astype("float32")

frac_snow_covered = float(binary.sum().values) / float(binary.notnull().sum().values)
print(f"Pixel fraction with depth > {DEPTH_THRESHOLD_MM:g} mm on {actual_day}: {frac_snow_covered:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
depth_day.plot(ax=axes[0], cmap="Blues", robust=True, cbar_kwargs={"orientation": "horizontal", "shrink": 0.8, "pad": 0.08, "aspect": 30, "label": "snow_depth (mm)"})
axes[0].set_title(f"snow_depth | {actual_day} | mm", fontsize=11)
axes[0].set_xlabel("x (m, EPSG:5070)")
axes[0].set_ylabel("y (m, EPSG:5070)")
axes[0].set_aspect("equal")

binary.plot(ax=axes[1], cmap="Blues", vmin=0, vmax=1, cbar_kwargs={"orientation": "horizontal", "shrink": 0.8, "pad": 0.08, "aspect": 30, "label": f"snow_depth > {DEPTH_THRESHOLD_MM:g} mm (0/1)"})
axes[1].set_title(f"binary | snow_depth > {DEPTH_THRESHOLD_MM:g} mm | {actual_day}", fontsize=11)
axes[1].set_xlabel("x (m, EPSG:5070)")
axes[1].set_ylabel("y (m, EPSG:5070)")
axes[1].set_aspect("equal")

fig.suptitle("Snow-depth → binary preview — PR-B aggregator pre-hook will area-weight this to HRUs", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Aggregator commute property** — *the* reason this transformation belongs pre-aggregation:

- Per-pixel: `depth_p > 1 mm` is a Boolean indicator on the 4 km grid.
- HRU area-weighted mean of the indicator = fraction of HRU area where depth > 1 mm = SCA.

Doing this on the HRU-mean depth (post-aggregation) would give the wrong answer: a 50%-snow-covered HRU with 1 cm snow on half its area would average to 5 mm depth, then `5 mm > 1 mm` = True → claim 100% SCA. The pixel-then-area-weight order is the only one that gives the right interpretation.

CLAUDE.md `Aggregation Transformation Policy` documents this — UA SWE is a textbook case for the pre-aggregate hook.

## 5. Seasonal cycle — a Western US sample box

Pick a 100×100-pixel box in the central Sierra Nevada (~`x=-1900000, y=2050000` in EPSG:5070), compute the box-mean SWE over the whole water year. Expect a typical accumulation/peak/melt curve with peak in March/April.

In [ ]:
# Sierra Nevada-ish bounding box in EPSG:5070 (approximate; doesn't need to be exact).
# Centre at roughly 38.5N, -119.5W which projects to about x=-1.94e6, y=2.05e6.
sierra_box = ds.sel(x=slice(-2050000, -1750000), y=slice(2200000, 1900000))
sierra_swe_mean = sierra_box["swe"].mean(dim=("x", "y"), skipna=True)
sierra_depth_mean = sierra_box["snow_depth"].mean(dim=("x", "y"), skipna=True)

fig, ax_swe = plt.subplots(figsize=(12, 5))
ax_swe.plot(sierra_swe_mean.time, sierra_swe_mean.values, color="tab:blue", label="SWE (mm = kg m-2)")
ax_swe.set_ylabel("SWE (mm)", color="tab:blue")
ax_swe.tick_params(axis="y", labelcolor="tab:blue")
ax_swe.set_xlabel("date")
ax_swe.set_title(f"Sierra Nevada bbox seasonal cycle — UA SWE WY{TARGET_WY}\nbox: x in [-2.05e6, -1.75e6], y in [1.90e6, 2.20e6] (EPSG:5070, ~300x300 km)", fontsize=11)

ax_depth = ax_swe.twinx()
ax_depth.plot(sierra_depth_mean.time, sierra_depth_mean.values, color="tab:orange", alpha=0.7, label="snow_depth (mm)")
ax_depth.set_ylabel("snow depth (mm)", color="tab:orange")
ax_depth.tick_params(axis="y", labelcolor="tab:orange")

# Combine legends from both axes.
lines_swe, labels_swe = ax_swe.get_legend_handles_labels()
lines_depth, labels_depth = ax_depth.get_legend_handles_labels()
ax_swe.legend(lines_swe + lines_depth, labels_swe + labels_depth, loc="upper right")

plt.tight_layout()
plt.show()

print(f"\nSierra bbox WY{TARGET_WY} peak SWE: {float(sierra_swe_mean.max()):.0f} mm on {str(sierra_swe_mean.idxmax().values)[:10]}")
print(f"Sierra bbox WY{TARGET_WY} peak depth: {float(sierra_depth_mean.max()):.0f} mm on {str(sierra_depth_mean.idxmax().values)[:10]}")
print(f"Sierra bbox WY{TARGET_WY} mean density at peak: SWE/depth = {float(sierra_swe_mean.max()) / float(sierra_depth_mean.max()):.2f} (kg/m^3 / 1000 = approx 0.3 - 0.4)")

**Expected pattern** for a normal Sierra winter:

- SWE accumulates Nov-Mar, peaks late March-early April, melts out by July.
- Snow depth follows the same envelope, scaled by snowpack density.
- Density (SWE / depth) climbs from ~0.15 (fresh fluffy snow) to ~0.4-0.5 (settled snowpack) by peak.
- Both curves return to 0 by mid-summer.

**Red flags:** flat curve (decoding bug), peak in October (timezone or WY-confusion bug), values dropping mid-winter (gap-fill failure).

## 6. Magnitude validation against Broxton CONUS climatology

Broxton et al. (2019, doi:[10.1175/JHM-D-19-0042.1](https://doi.org/10.1175/JHM-D-19-0042.1)) report mean-CONUS-SWE-on-March-1 of roughly **30-50 mm** averaged across their study period (1981-2017). For any single year this can swing 50-100% either way depending on snowpack anomaly. **Order-of-magnitude wrong** here would be a smoking gun for a missed conversion factor — order-of-magnitude correct is what we need to see.

In [ ]:
# Mean-CONUS SWE on the target day (already loaded above as target_day).
conus_mean_swe = float(target_day["swe"].mean(skipna=True).values)
conus_max_swe = float(target_day["swe"].max(skipna=True).values)
conus_finite_swe = float(target_day["swe"].notnull().sum().values)
print(f"WY{TARGET_WY} {actual_day} CONUS-mean SWE (over finite pixels): {conus_mean_swe:.1f} mm")
print(f"WY{TARGET_WY} {actual_day} CONUS-max  SWE (single pixel)       : {conus_max_swe:.0f} mm")
print(f"WY{TARGET_WY} {actual_day} finite-pixel count                 : {int(conus_finite_swe):,}")
print()
print("Broxton et al. (2019) CONUS-mean Mar-1 climatology (1981-2017): ~30 - 50 mm")
print(f"This-year ratio to climatology midpoint (40 mm): {conus_mean_swe / 40.0:.2f}x")
print()
if 10 <= conus_mean_swe <= 100:
    print("Magnitude OK (within plausible inter-annual range).")
else:
    print("RED FLAG: out of plausible range — investigate decoding / units.")

## 7. Cross-WY consistency — pick a year per decade

Not every consolidated WY can be in one notebook — but a quick "pick one peak day per decade" scan helps catch a regression introduced by re-running the consolidator. Loops cheap: just open the WY file, pull March 1, compute CONUS mean.

In [ ]:
scan_wys = [1985, 1995, 2005, 2015, 2023]
print(f"{'WY':>4} {'date':>12} {'CONUS-mean SWE (mm)':>22} {'snow_depth mean (mm)':>22}")
print("-" * 64)
for wy in scan_wys:
    p = DAILY_DIR / f"ua_swe_daily_WY{wy}.nc"
    if not p.exists():
        print(f"{wy:>4} {'(missing)':>12}")
        continue
    with xr.open_dataset(p) as scan_ds:
        target = f"{wy}-03-01"
        if target not in scan_ds.time.values.astype("datetime64[D]").astype(str):
            target = scan_ds.time.sel(time=target, method="nearest").values
            target = str(target)[:10]
        day = scan_ds.sel(time=target, method="nearest")
        swe_mean = float(day["swe"].mean(skipna=True).values)
        depth_mean = float(day["snow_depth"].mean(skipna=True).values)
        print(f"{wy:>4} {target:>12} {swe_mean:>22.1f} {depth_mean:>22.1f}")
print()
print("All five should fall in the 10 - 100 mm SWE band; outside that, investigate that WY's consolidator output.")

## Summary — proceed or not?

If everything above shows:

1. ✅ Schema matches the catalog (`swe` kg m-2, `snow_depth` mm, `cell_methods: time: point`).
2. ✅ Peak-winter map shows the expected Sierra / Cascades / Rockies / Adirondacks signal.
3. ✅ CONUS land-footprint coverage is ~50-60% of the EPSG:5070 box (expected — ocean and non-land masked out).
4. ✅ Sierra bbox shows a normal accumulation/peak/melt curve.
5. ✅ CONUS-mean March-1 SWE is in the 10 - 100 mm range.
6. ✅ Cross-WY scan shows similar magnitudes across decades.

…then UA SWE is wired correctly at the consolidator layer and we should proceed with **PR-B (aggregate)**, **PR-C (SWE target)**, and **PR-D (multi-source SCA)** per the umbrella plan.

## Clean up

In [ ]:
ds.close()